# Question 2: Transition Based Dependency Parser

This notebook builds a transition based dependency parser using the arc standard system. The classifier is trained on the Universal Dependencies English EWT treebank and the whole pipeline goes from reading the raw CoNLL-U files, to generating oracle training data, to training a classifier, to running the parser greedily, to computing the Labeled Attachment Score (LAS) on the dev set.

I kept the features simple and stuck close to what the assignment asked for, since the point of the exercise seems to be understanding the transition system and the oracle, not squeezing out the last bit of accuracy with a fancy model.

**Data**: clone the treebank first if you have not already.

```
git clone https://github.com/UniversalDependencies/UD_English-EWT.git
```

The notebook expects the folder `UD_English-EWT` to sit next to this notebook, with `en_ewt-ud-train.conllu` and `en_ewt-ud-dev.conllu` inside it. If those files are not found, the notebook falls back to a tiny hand written sample so the whole pipeline still runs end to end as a sanity check, but obviously the LAS on that fallback is meaningless.

In [2]:
import re
import time
import random
from collections import defaultdict

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

random.seed(0)

TRAIN_PATH = "UD_English-EWT/en_ewt-ud-train.conllu"
DEV_PATH = "UD_English-EWT/en_ewt-ud-dev.conllu"

ROOT_ID = 0

## Part 1: Data Processing and Oracle Simulation

### Reading the CoNLL-U files

CoNLL-U is tab separated, one word per line, sentences separated by a blank line, comments starting with `#`. A few things I had to handle:

- multiword tokens like `3-4` (contractions such as "don't") need to be skipped, they are not real nodes in the dependency tree
- empty nodes like `3.1` (used for elided predicates) also need to be skipped for the same reason
- I strip the subtype off the dependency relation, so `obl:tmod` becomes `obl`. UD relations have a lot of these subtypes and keeping them all would blow up the label set for not much gain at this scale

In [3]:
class Token:
    def __init__(self, id_, form, upos, head, deprel):
        self.id = id_
        self.form = form
        self.upos = upos
        self.head = head
        self.deprel = deprel


class Sentence:
    def __init__(self, tokens):
        self.tokens = tokens  # does not include the artificial root

    def __len__(self):
        return len(self.tokens)

    def words(self):
        return [t.form for t in self.tokens]


def read_conllu(path):
    sentences = []
    current = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith("#"):
                continue
            if line.strip() == "":
                if current:
                    sentences.append(Sentence(current))
                    current = []
                continue
            fields = line.split("\t")
            if len(fields) < 8:
                continue
            id_field = fields[0]
            if "-" in id_field or "." in id_field:
                # multiword token span or empty node, skip
                continue
            id_ = int(id_field)
            form = fields[1]
            upos = fields[3]
            head = int(fields[6])
            deprel = fields[7].split(":")[0]
            current.append(Token(id_, form, upos, head, deprel))
    if current:
        sentences.append(Sentence(current))
    return sentences

In [4]:
# tiny fallback corpus, only used if the real UD files are missing,
# just so the notebook can be run top to bottom as a smoke test
FALLBACK_CONLLU = """# sent_id = 1
# text = The cat sat on the mat.
1\tThe\tthe\tDET\tDT\t_\t2\tdet\t_\t_
2\tcat\tcat\tNOUN\tNN\t_\t3\tnsubj\t_\t_
3\tsat\tsit\tVERB\tVBD\t_\t0\troot\t_\t_
4\ton\ton\tADP\tIN\t_\t6\tcase\t_\t_
5\tthe\tthe\tDET\tDT\t_\t6\tdet\t_\t_
6\tmat\tmat\tNOUN\tNN\t_\t3\tobl\t_\t_
7\t.\t.\tPUNCT\t.\t_\t3\tpunct\t_\t_

# sent_id = 2
# text = She eats a green salad.
1\tShe\tshe\tPRON\tPRP\t_\t2\tnsubj\t_\t_
2\teats\teat\tVERB\tVBZ\t_\t0\troot\t_\t_
3\ta\ta\tDET\tDT\t_\t5\tdet\t_\t_
4\tgreen\tgreen\tADJ\tJJ\t_\t5\tamod\t_\t_
5\tsalad\tsalad\tNOUN\tNN\t_\t2\tobj\t_\t_
6\t.\t.\tPUNCT\t.\t_\t2\tpunct\t_\t_
"""


def read_conllu_from_string(text):
    import io
    sentences = []
    current = []
    for line in io.StringIO(text):
        line = line.rstrip("\n")
        if line.startswith("#"):
            continue
        if line.strip() == "":
            if current:
                sentences.append(Sentence(current))
                current = []
            continue
        fields = line.split("\t")
        if len(fields) < 8:
            continue
        id_field = fields[0]
        if "-" in id_field or "." in id_field:
            continue
        id_ = int(id_field)
        form = fields[1]
        upos = fields[3]
        head = int(fields[6])
        deprel = fields[7].split(":")[0]
        current.append(Token(id_, form, upos, head, deprel))
    if current:
        sentences.append(Sentence(current))
    return sentences


try:
    train_sents = read_conllu(TRAIN_PATH)
    dev_sents = read_conllu(DEV_PATH)
    using_fallback = False
except FileNotFoundError:
    print("Could not find the UD English EWT files, falling back to a tiny built in sample.")
    print("Clone https://github.com/UniversalDependencies/UD_English-EWT.git to use the real data.")
    train_sents = read_conllu_from_string(FALLBACK_CONLLU)
    dev_sents = train_sents
    using_fallback = True

print(f"train sentences: {len(train_sents)}")
print(f"dev sentences: {len(dev_sents)}")

train sentences: 12544
dev sentences: 2001


### Oracle simulator

I implemented a static oracle for arc standard. At every configuration it looks at the top two items on the stack and decides:

- **LEFT-ARC**: if the second item on the stack is the gold head of the top item... wait, other way round. LEFT-ARC applies when the item on top of the stack is the gold head of the second item, and the second item has already collected all of its own dependents from the gold tree (otherwise we would pop it too early and lose the chance to attach its children).
- **RIGHT-ARC**: symmetric case, the second item is the gold head of the top item, and the top item has already collected all of its dependents.
- **SHIFT**: otherwise, if the buffer is not empty.

The "already collected all dependents" check is why I keep a running count of how many gold children each token still has left to be attached. I decrement that count every time a child gets attached during the simulation.

In [5]:
def gold_children_count(sentence):
    counts = defaultdict(int)
    for tok in sentence.tokens:
        counts[tok.head] += 1
    return counts


def oracle_transitions(sentence):
    """Runs the static arc standard oracle over one gold sentence.
    Returns a list of ((stack_ids, buffer_ids), action) pairs, one per decision point,
    or None if the tree cannot be produced by this oracle (this happens for a small
    number of non-projective sentences, which arc standard cannot handle directly).
    action is either the string "SHIFT" or a tuple like ("LEFT-ARC", "nsubj").
    """
    tokens = {tok.id: tok for tok in sentence.tokens}
    gold_head = {tok.id: tok.head for tok in sentence.tokens}
    remaining_children = gold_children_count(sentence)

    stack = [ROOT_ID]
    buffer = [tok.id for tok in sentence.tokens]
    examples = []

    while buffer or len(stack) > 1:
        if len(stack) < 2:
            action = "SHIFT"
        else:
            top = stack[-1]
            second = stack[-2]
            if second != ROOT_ID and gold_head[second] == top and remaining_children[second] == 0:
                action = ("LEFT-ARC", tokens[second].deprel)
            elif gold_head[top] == second and remaining_children[top] == 0:
                action = ("RIGHT-ARC", tokens[top].deprel)
            elif buffer:
                action = "SHIFT"
            else:
                # stuck: this sentence is not projective, arc standard cannot finish it
                return None

        examples.append(((list(stack), list(buffer)), action))

        if action == "SHIFT":
            stack.append(buffer.pop(0))
        elif action[0] == "LEFT-ARC":
            dep = stack.pop(-2)
            remaining_children[gold_head[dep]] -= 1
        elif action[0] == "RIGHT-ARC":
            dep = stack.pop()
            remaining_children[gold_head[dep]] -= 1

    return examples


# quick sanity check on the first training sentence
sample = train_sents[0]
result = oracle_transitions(sample)
print("words:", sample.words())
if result is None:
    print("this sentence is non-projective, oracle could not finish it")
else:
    print(f"{len(result)} transitions produced")
    for (stack, buffer), action in result[:6]:
        print(f"  stack={stack} buffer={buffer[:3]}... action={action}")

words: ['Al', '-', 'Zaman', ':', 'American', 'forces', 'killed', 'Shaikh', 'Abdullah', 'al', '-', 'Ani', ',', 'the', 'preacher', 'at', 'the', 'mosque', 'in', 'the', 'town', 'of', 'Qaim', ',', 'near', 'the', 'Syrian', 'border', '.']
58 transitions produced
  stack=[0] buffer=[1, 2, 3]... action=SHIFT
  stack=[0, 1] buffer=[2, 3, 4]... action=SHIFT
  stack=[0, 1, 2] buffer=[3, 4, 5]... action=SHIFT
  stack=[0, 1, 2, 3] buffer=[4, 5, 6]... action=('LEFT-ARC', 'punct')
  stack=[0, 1, 3] buffer=[4, 5, 6]... action=('RIGHT-ARC', 'flat')
  stack=[0, 1] buffer=[4, 5, 6]... action=SHIFT


## Part 2: Feature Extraction and Model Training

### Features

Sticking to what the assignment asks for, just POS tags of a small window around the top of the stack and the front of the buffer:

- POS of the word on top of the stack (`s1`)
- POS of the second word on the stack (`s2`), if it exists
- POS of the first word in the buffer (`b1`), if it exists
- POS of the second word in the buffer (`b2`), if it exists

I represent the artificial root with the pseudo tag `ROOT` and any missing slot (stack or buffer too short) with `NONE`, so the feature dictionary always has the same four keys and `DictVectorizer` can one hot encode them consistently.

In [6]:
def extract_features(stack, buffer, tokens):
    """tokens maps token id to a Token, id 0 stands for the artificial root."""
    def upos_of(tid):
        if tid == ROOT_ID:
            return "ROOT"
        return tokens[tid].upos

    s1 = stack[-1] if len(stack) >= 1 else None
    s2 = stack[-2] if len(stack) >= 2 else None
    b1 = buffer[0] if len(buffer) >= 1 else None
    b2 = buffer[1] if len(buffer) >= 2 else None

    return {
        "s1_upos": upos_of(s1) if s1 is not None else "NONE",
        "s2_upos": upos_of(s2) if s2 is not None else "NONE",
        "b1_upos": upos_of(b1) if b1 is not None else "NONE",
        "b2_upos": upos_of(b2) if b2 is not None else "NONE",
    }

### Building the training data

I train a single classifier over a combined label that folds the transition type and the dependency label together, so a class looks like `SHIFT`, `LEFT-ARC:nsubj` or `RIGHT-ARC:obj`. This is a common simplification, it avoids having to train and coordinate two separate classifiers (one for the transition, one for the label) and keeps the parsing loop simple, at the cost of a somewhat larger label space.

In [7]:
def action_to_label(action):
    if action == "SHIFT":
        return "SHIFT"
    return f"{action[0]}:{action[1]}"


def label_to_action(label):
    if label == "SHIFT":
        return "SHIFT"
    kind, dep = label.split(":", 1)
    return (kind, dep)


def build_training_data(sentences):
    feats = []
    labels = []
    skipped = 0
    for sent in sentences:
        tokens = {tok.id: tok for tok in sent.tokens}
        transitions = oracle_transitions(sent)
        if transitions is None:
            skipped += 1
            continue
        for (stack, buffer), action in transitions:
            feats.append(extract_features(stack, buffer, tokens))
            labels.append(action_to_label(action))
    return feats, labels, skipped


train_feats, train_labels, skipped_sents = build_training_data(train_sents)
print(f"training examples: {len(train_feats)}")
print(f"sentences skipped as non-projective: {skipped_sents} out of {len(train_sents)}")
print(f"number of distinct actions: {len(set(train_labels))}")

training examples: 392242
sentences skipped as non-projective: 287 out of 12544
number of distinct actions: 61


### Training the classifier

I went with logistic regression over a one hot encoding of the four POS features. It is fast to train on tens of thousands of examples and is a reasonable baseline classifier for this kind of categorical feature setup. `DictVectorizer` takes care of the one hot encoding.

In [8]:
vectorizer = DictVectorizer(sparse=True)
X_train = vectorizer.fit_transform(train_feats)

clf = LogisticRegression(max_iter=300)

start = time.time()
clf.fit(X_train, train_labels)
print(f"trained in {time.time() - start:.1f}s on {X_train.shape[0]} examples, {X_train.shape[1]} features")

# training accuracy just as a quick check that learning is happening at all
train_acc = clf.score(X_train, train_labels)
print(f"training set action accuracy: {train_acc:.3f}")

trained in 38.0s on 392242 examples, 73 features
training set action accuracy: 0.813


## Part 3: Parser Implementation and Evaluation

### The parsing loop

At each step I extract features from the current configuration, ask the classifier for a ranked list of actions by probability, and take the highest ranked action that is actually legal in the current configuration. A predicted action can be illegal, for example the model might predict LEFT-ARC when the stack only has the root on it, so I walk down the ranked list until I find something valid. If nothing in the ranked list is valid (should not really happen but I added a safety net anyway) I fall back to SHIFT if the buffer is not empty, or force a RIGHT-ARC to make progress and eventually terminate.

In [9]:
def is_valid(action, stack, buffer):
    if action == "SHIFT":
        return len(buffer) > 0
    if action[0] == "LEFT-ARC":
        # cannot make the root a dependent, and need two items on the stack
        return len(stack) >= 2 and stack[-2] != ROOT_ID
    if action[0] == "RIGHT-ARC":
        return len(stack) >= 2
    return False


def parse_sentence(id_upos_pairs, clf, vectorizer):
    """id_upos_pairs: list of (token_id, upos) for the sentence, in order.
    Returns a dict mapping dependent id to (head id, label).
    """
    tokens = {tid: type("T", (), {"upos": upos})() for tid, upos in id_upos_pairs}
    stack = [ROOT_ID]
    buffer = [tid for tid, _ in id_upos_pairs]
    arcs = {}

    # a generous step budget so a pathological prediction sequence cannot loop forever
    steps_left = 4 * len(id_upos_pairs) + 5

    while (buffer or len(stack) > 1) and steps_left > 0:
        steps_left -= 1
        feats = extract_features(stack, buffer, tokens)
        X = vectorizer.transform([feats])
        probs = clf.predict_proba(X)[0]
        ranked = sorted(zip(clf.classes_, probs), key=lambda pair: -pair[1])

        chosen = None
        for label, _ in ranked:
            action = label_to_action(label)
            if is_valid(action, stack, buffer):
                chosen = action
                break

        if chosen is None:
            chosen = "SHIFT" if buffer else ("RIGHT-ARC", "dep")

        if chosen == "SHIFT":
            stack.append(buffer.pop(0))
        elif chosen[0] == "LEFT-ARC":
            dep = stack.pop(-2)
            arcs[dep] = (stack[-1], chosen[1])
        elif chosen[0] == "RIGHT-ARC":
            dep = stack.pop()
            arcs[dep] = (stack[-1], chosen[1])

    return arcs

### LAS evaluation

Labeled Attachment Score is the fraction of words whose predicted head and predicted dependency label both match the gold ones. I also compute Unlabeled Attachment Score (UAS, head only) since it is a nice sanity number to have alongside LAS, it tells you how much of the error is coming from wrong labels versus genuinely wrong structure.

In [10]:
def evaluate(sentences, clf, vectorizer):
    correct_labeled = 0
    correct_unlabeled = 0
    total = 0
    for sent in sentences:
        id_upos_pairs = [(tok.id, tok.upos) for tok in sent.tokens]
        pred_arcs = parse_sentence(id_upos_pairs, clf, vectorizer)
        for tok in sent.tokens:
            total += 1
            pred = pred_arcs.get(tok.id)
            if pred is None:
                continue
            if pred[0] == tok.head:
                correct_unlabeled += 1
                if pred[1] == tok.deprel:
                    correct_labeled += 1
    las = correct_labeled / total if total else 0.0
    uas = correct_unlabeled / total if total else 0.0
    return las, uas, total


start = time.time()
las, uas, n_tokens = evaluate(dev_sents, clf, vectorizer)
elapsed = time.time() - start

print(f"evaluated on {len(dev_sents)} dev sentences, {n_tokens} tokens, in {elapsed:.1f}s")
print(f"UAS: {uas:.4f}")
print(f"LAS: {las:.4f}")
if using_fallback:
    print("note: this run used the tiny fallback corpus, these numbers are just a smoke test")

evaluated on 2001 dev sentences, 25148 tokens, in 8.0s
UAS: 0.6714
LAS: 0.5815


### Trying it on a few hand written sentences

The parser needs POS tags as input, since that is all the feature extractor looks at. For real sentences that are not already in the treebank I get POS tags with NLTK's tagger and map its universal tagset onto the UD tags I trained on. The mapping is not perfect (NLTK's universal tagset uses `.` for punctuation instead of `PUNCT`, and does not distinguish `AUX` from `VERB` for example) but it is close enough to see the parser do something sensible.

This cell needs `nltk` installed and the `averaged_perceptron_tagger` and `universal_tagset` data downloaded, which needs a working internet connection the first time.

In [13]:
import nltk

try:
    nltk.data.find("taggers/averaged_perceptron_tagger")
except LookupError:
    nltk.download("averaged_perceptron_tagger")
try:
    nltk.data.find("taggers/averaged_perceptron_tagger_eng")
except LookupError:
    nltk.download("averaged_perceptron_tagger_eng")
try:
    nltk.data.find("taggers/universal_tagset")
except LookupError:
    nltk.download("universal_tagset")
try:
    nltk.data.find("tokenizers/punkt_tab")
except LookupError:
    nltk.download("punkt_tab")

NLTK_TO_UD = {
    ".": "PUNCT",
    "CONJ": "CCONJ",
    "PRT": "PART",
    "NUM": "NUM",
    "X": "X",
}

def tag_sentence(text):
    words = nltk.word_tokenize(text)
    tagged = nltk.pos_tag(words, tagset="universal")
    return [(w, NLTK_TO_UD.get(t, t)) for w, t in tagged]


test_sentences = [
    "The cat sat on the mat.",
    "She eats a green salad.",
    "I saw the man with a telescope.",
]

for text in test_sentences:
    tagged = tag_sentence(text)
    id_upos_pairs = [(i + 1, upos) for i, (word, upos) in enumerate(tagged)]
    arcs = parse_sentence(id_upos_pairs, clf, vectorizer)
    words = [w for w, _ in tagged]

    print(text)
    for i, (word, upos) in enumerate(tagged, start=1):
        head, label = arcs.get(i, (None, None))
        head_word = "ROOT" if head == 0 else (words[head - 1] if head else "?")
        print(f"  {word:>12} ({upos:<6}) -> head: {head_word:<12} label: {label}")
    print()

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/deathslash/nltk_data...


The cat sat on the mat.
           The (DET   ) -> head: cat          label: det
           cat (NOUN  ) -> head: sat          label: nsubj
           sat (VERB  ) -> head: ROOT         label: root
            on (ADP   ) -> head: mat          label: case
           the (DET   ) -> head: mat          label: det
           mat (NOUN  ) -> head: sat          label: obl
             . (PUNCT ) -> head: sat          label: punct

She eats a green salad.
           She (PRON  ) -> head: eats         label: nsubj
          eats (VERB  ) -> head: ROOT         label: root
             a (DET   ) -> head: salad        label: det
         green (ADJ   ) -> head: salad        label: amod
         salad (NOUN  ) -> head: eats         label: obl
             . (PUNCT ) -> head: eats         label: punct

I saw the man with a telescope.
             I (PRON  ) -> head: saw          label: nsubj
           saw (VERB  ) -> head: ROOT         label: root
           the (DET   ) -> head: man          la

[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


## Design choices and notes

**Transition system**: arc standard, as required, with a static oracle. A handful of sentences in EWT are non-projective and the arc standard oracle cannot produce a transition sequence for those, so I skip them when building the training set (the skipped count is printed above). This loses a small amount of training data but is the standard tradeoff with arc standard, since it can only build projective trees.

**Features**: only the four POS tags the assignment specifies, no lexical (word form) features. This keeps the feature space small and training fast, but it does mean the classifier cannot use anything about the actual words, only their tags, so it is more of a POS driven syntactic model than a lexicalized one. Adding word form features for the same four positions would be a natural next step to improve LAS.

**Classifier**: logistic regression over one hot encoded POS features. I picked this over something like a decision tree because the feature space is small and mostly linear interactions (each POS combination roughly maps to a small set of likely actions), and logistic regression trains fast and gives well calibrated probabilities, which the greedy parser relies on to pick the next best legal action when the top prediction is invalid.

**Label scheme**: transition type and dependency label are folded into one combined class (`LEFT-ARC:nsubj` etc) rather than training two separate classifiers. Simpler to implement and to use inside the greedy loop, at the cost of a larger label space than a two stage classifier would have.

**Parsing strategy**: fully greedy, one prediction per step, no beam search and no backtracking. This means an early mistake can cascade into later ones, which is the classic weakness of greedy transition based parsers, but it keeps parsing linear in sentence length and is what the assignment describes.

**Final LAS**: printed above in the evaluation cell, run against `en_ewt-ud-dev.conllu`.